## Cell 3: Imports

In [2]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from langchain_ollama import ChatOllama
from langchain_core.tools import tool

import sqlite3
import json

## cell 4: Initialize Ollama LLM

In [3]:
llm = ChatOllama(
    model = "llama3.2",
    temperature = 0
)

## Cell 5: Long-Term Memory Setup (SQLite)

In [5]:
conn = sqlite3.connect("memory.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS memory(
    user_id TEXT,
    key TEXT,
    value TEXT
)
""")

conn.commit()

## Cell 6: Memory Function

In [6]:
def save_memory(user_id, key, value):
    cursor.execute(
        "INSERT INTO memory VALUES (?, ?, ?)",
        (user_id, key, json.dumps(value))
    )
    conn.commit()

def load_memory(user_id):
    cursor.execute(
        "SELECT key, value FROM memory WHERE user_id=?",
        (user_id,)
    )
    rows = cursor.fetchall()

    memory = {}
    for k, v in rows:
        memory[k] = json.loads(v)
    
    return memory

## Cell 7: DEFINE TOOLS

### Calculator Tool

In [7]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

### Memory Saver Tool

In [8]:
@tool 
def remember(user_id: str, key: str, value: str) -> str:
    """save user info."""
    save_memory(user_id, key, value)
    return f"Saved {key} = {value}"

### Memory Recall Tool

In [9]:
@tool
def recall(user_id: str, key: str) -> str:
    """Retrieve user info."""
    memory = load_memory(user_id)
    return str(memory.get(key, "Not found"))

## Cell 8: Bind Tools to LLM

In [10]:
tools = [calculator, remember, recall]
llm_with_tools = llm.bind_tools(tools)

## Cell 9: Agent Node

In [12]:
def agent_node(state: MessagesState, config):
    user_id = config["configurable"]["thread_id"]

    messages = state["messages"]

    # Load long-term memory
    memory = load_memory(user_id)

    # System prompt with memory
    system_msg = (
        "you are a smart assistant.\n"
        "Use tools when needed.\n"
        f"User info: {memory}"
    )

    final_message = [("system", system_msg), *messages]

    response = llm_with_tools.invoke(final_messages)

    return {"messages": [response]}

## Cell 10: Tool Execution Node

In [14]:
def tool_node(state: MessagesState, config):
    last_message = state["messages"][-1]
    
    tool_calls = last_message.tool_calls

    results = []

    for call in tool_calls:
        tool_name = call["name"]
        arge = call["args"]

        if tool_name == "calculator":
            result = calcualtor.invoke(args)
        elif tool_name == "remember":
            result = remember.invoke(args)
        elif tool_name == "recall":
            result = recall.invoke(args)

        results.append({
            "role": "tool",
            "content":result,
            "tool_call_id":call["id"]
        })

    return {"messages":results}